# [실습] 라벨 검수표 — 불만 비율 보정하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 질문과 데이터

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">비정형 데이터(자연어) · 골드 표본과 LLM 라벨 검증</div>
<h1 style="color:#000000;margin:10px 0">라벨 검수표 — 불만 비율 보정하기</h1>
<div style="font-size:1.05em;color:#656565">리뷰 30건을 직접 라벨링해 LLM 라벨과 비교하고, 내 라벨로 리뷰 1,000건의 불만 비율을 보정합니다. 별점이 있는 표본의 크기에 따라 구간이 얼마나 짧아지는지도 확인합니다</div>
</div>

**핵심 질문** LLM 라벨로 계산한 비율을 그대로 써도 될까요?

이론에서 배운 골드 표본 비교와 비율 보정을 실제 쇼핑 리뷰로 직접 해 봅니다. 계산하는 코드는 모두 준비되어 있습니다. 출력을 읽고 판단을 적습니다.

1. **골드 표본 30건 라벨링하기**: 리뷰 30건을 직접 라벨링하고 LLM 라벨과의 일치율 · κ 를 구합니다.
2. **κ 가 높으면 비율도 맞는가**: LLM 만 있음과 나만 있음의 건수로 편향을 알아보고, 리뷰 2만 건의 부정 비율을 구하는 방법을 정합니다.
3. **보정한 비율**: 내 라벨로 리뷰 1,000건의 불만 비율을 보정하고 결과 문장을 씁니다.
4. **표본 크기**: 별점이 있는 표본 30 · 120 · 480건의 구간 길이로 필요한 건수를 정합니다.
5. **집단 · 범주마다 다른 재현율**: 리뷰 길이 두 집단의 차이와 드문 상담이유의 건수를 라벨로 구해도 되는지 판단합니다.

단계마다 설명을 읽고, 코드를 실행해 출력을 읽은 뒤, 직접 해 보기를 채웁니다. 직접 해 보기에서는 `MY_` 로 시작하는 변수를 채웁니다. 바로 아래 확인 셀을 실행하면 결과가 나옵니다.

**실행 안내**

- LMS 에서는 셀을 위에서부터 차례로 실행합니다. 이 실습은 모델을 호출하지 않아 GPU 가 필요 없습니다.
- Colab 이나 내 컴퓨터에서 열었다면 맨 위 설치 셀부터 실행합니다. 처음 한 번은 원자료를 받고 분류기를 학습하느라 1분 안팎 걸립니다.
- 확인 셀은 변수를 아직 채우지 않아도 오류 없이 안내만 출력합니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f7/NachoKamenov-Humansin_the_LoopDataannotators_discussing_the_correct_labeling_of_a_dataset-3543x2362.jpg/960px-NachoKamenov-Humansin_the_LoopDataannotators_discussing_the_correct_labeling_of_a_dataset-3543x2362.jpg" alt="두 사람이 모니터 앞에 함께 앉아 모니터 하나를 보며 라벨을 두고 이야기하는 사진" width="560" height="373"/>
<p align="center"><em>▲ 데이터 라벨러 두 사람이 모니터 하나를 보며 라벨을 맞추는 모습. 같은 리뷰도 사람마다 라벨이 다를 수 있어, 두 사람이 따로 라벨링한 뒤 비교한다 (출처: Wikimedia Commons, Nacho Kamenov & Humans in the Loop, CC BY 4.0)</em></p>

In [ ]:
# 실습 준비: 설치·폰트. 폰트 설치는 Colab 에서만 합니다
#  이 실습은 새로 설치하는 패키지가 없습니다.
#  numpy · pandas · scikit-learn · pyarrow · matplotlib 은 Colab 기본 버전을 그대로 씁니다.
#  본문의 수는 scikit-learn 1.8.0 · pandas 2.3.3 · numpy 2.4.2 에서 계산했습니다. 버전이 다르면 아래에 한 행으로 알립니다.
#  한국어 형태소 분석기 kiwipiepy 0.23.1 은 「자료 준비」 셀이 민원 사건제목의 명사 파일(과정 결과)을 받지 못했을 때만 그 셀에서 설치합니다.
#  맨 아래에서 확인 셀이 쓰는 답 확인 도우미(ac_ 로 시작하는 함수)를 정의합니다. LMS 에서도 이 셀을 먼저 실행합니다.
import os, sys, math, time
import subprocess
IN_COLAB = "google.colab" in sys.modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트: Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  등록해야 이름이 보인다(등록하지 않으면 DejaVu Sans 로 바뀌어 한글이 전부 □ 로 나온다).
import glob, shutil
if IN_COLAB and shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 이 실습의 그림 전체가 함께 쓰는 색: 기준선=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 이 실습의 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다

from importlib.metadata import version as _ver, PackageNotFoundError
_want = {"scikit-learn": "1.8.0", "pandas": "2.3.3", "numpy": "2.4.2"}
_have = {}
for _name in _want:
    try:
        _have[_name] = _ver(_name)
    except PackageNotFoundError:
        _have[_name] = "없음"
print("사용 폰트:", FONT, "| 색 팔레트 준비 완료 |", " · ".join(f"{k} {v}" for k, v in _have.items()))
_diff = [k for k in _want if _have[k] != _want[k]]
if _diff:
    print("참고: 본문의 수는 위 버전에서 계산했습니다. 버전이 다른 패키지(" + " · ".join(_diff)
          + ")가 있어 단계 5 의 재현율이 한 건 차이로 달라질 수 있습니다.")

# >>> answer_check: 정본은 orchestration/lesson_build/practice_kit/answer_check.py 이고 손으로 고치지 않는다
# 노트북 방식 실습의 확인 셀이 함께 쓰는 도우미. 학생이 채운 값의 상태(비었는지 · 형식)와 글 답의 요소(수 · 원인 주장 · 단어)를
# 같은 규칙으로 본다. 실습마다 따로 짜던 정규식에서 같은 버그가 되풀이되었다(건수만 있는 결론을 수가 있다고 보거나,
# 「볼 수 없」 · 「원인을 밝히려면」을 원인 주장으로 잡음). 고칠 때는 이 정본을 고치고 embed_kit.py 로 노트북에 다시 반영한다.
import re as _re

AC_NEG = _re.compile(r"수\s?(?:는\s)?없|모릅|모른|모르|확인하지\s?못|아닙|아니다|아니라|않습|않는다|않았|않으|는지|실험|추가\s?분석|"
                     r"밝히려면|살펴야|확인해야")
AC_CLAUSE = _re.compile(r"[!?。\n]|\.(?!\d)|(?<!\d)\.|(?<=아니라)|(?<=지만)|(?<=는데)")      # 「…아니라 · …지만」 뒤에서도 문장을 끊어, 부정이 그 앞부분에만 걸리게
AC_INTERVAL = r"\d+\s?%\s?(?:신뢰\s?)?구간"                               # 「N% 구간」의 N 은 결과 수가 아님
AC_COUNT = (r"\d[\d,]*\s?점", r"\d[\d,]*\s?건", r"\d[\d,]*\s?명", r"\d[\d,]*\s?개", r"\d{4}-\d{2}-\d{2}", r"\d{1,2}월", r"단계\s?\d")


def ac_state(value, kind="text"):
    """(채웠는가, 안내 문장 또는 None). kind: text · bool · number · list."""
    if value is None or (isinstance(value, str) and not value.strip()) or (isinstance(value, (list, tuple)) and not value):
        return False, "아직 채우지 않았습니다"
    if kind == "text" and not isinstance(value, str):
        return False, "고칠 것: 따옴표 안에 문장으로 적습니다"
    if kind == "bool" and not isinstance(value, bool):
        return False, "고칠 것: 따옴표 없이 True 나 False 로 적습니다"
    if kind == "number" and (isinstance(value, bool) or not isinstance(value, (int, float))):
        return False, "고칠 것: 따옴표 없이 수로 적습니다"
    if kind == "list" and not isinstance(value, (list, tuple)):
        return False, "고칠 것: 대괄호 [ ] 안에 쉼표로 이어 적습니다"
    return True, None


def ac_numbers(text, ignore=AC_COUNT):
    """글에 든 수(쉼표 · % 를 뗀 float). 건수 · 점수 · 날짜처럼 결과가 아닌 수는 ignore 로 뺀다."""
    t = str(text)
    for pat in ignore:
        t = _re.sub(pat, " ", t)
    return [float(x.replace(",", "")) for x in _re.findall(r"(?<![\w.])-?\d[\d,]*(?:\.\d+)?", t)]


def ac_has_value(text, value, digits=None, tol=None):
    """글에 value 가 있는가. digits 를 주면 그 자릿수로 반올림한 값과 비교하고, 아니면 tol(기본은 값의 1/200)."""
    nums = ac_numbers(text, ignore=())
    if digits is not None:
        return any(abs(v - round(value, digits)) < 10 ** -digits / 2 + 1e-9 for v in nums)
    tol = abs(value) * 0.005 if tol is None else tol
    return any(abs(v - value) <= tol for v in nums)


def ac_has_stat(text):
    """결과 수(소수 · N배 · a～b 범위)가 있는가. 건수 · 점수 · 날짜만 있으면 False."""
    t = str(text)
    for pat in AC_COUNT + (AC_INTERVAL,):
        t = _re.sub(pat, " ", t)
    return bool(_re.search(r"\d\.\d|\d\s?배|\d\s?[～~]\s?\d|\d\s?%", t))


def ac_claims(text, claim_re, neg=AC_NEG):
    """문장을 「…아니라 · …지만」 뒤에서도 끊어 부분마다 claim_re(예: 원인 주장 단어)를 찾는다. 부정하거나 확인할 일로 쓴 부분은 뺀다."""
    found = []
    for sent in AC_CLAUSE.split(str(text)):
        if sent.strip() and not neg.search(sent):
            found += [m.group(0) for m in claim_re.finditer(sent)]
    return sorted(set(found))


def ac_show(title, items, yes="있음", no="없음"):
    """점검표를 같은 형식으로 출력한다. items: {이름: 참거짓}. 반환값은 맞은 수."""
    k = sum(bool(v) for v in items.values())
    print(f"{title}: {len(items)}가지 가운데 {k}가지")
    for name, ok in items.items():
        print(f"  {name}: {yes if ok else no}")
    return k
# <<< answer_check

사용 폰트: Pretendard | 색 팔레트 준비 완료 | scikit-learn 1.8.0 · pandas 2.3.3 · numpy 2.4.2

## 자료 준비

리뷰 자료는 이론과 같고, 분석 단위는 리뷰 한 건입니다. 민원은 이론의 예보다 큰 표본으로 상담이유 분류기를 학습합니다.

| 자료 | 이 실습에서 쓰는 것 |
|---|---|
| 네이버 쇼핑 리뷰(bab2min/corpus, 별점 1·2·4·5점) | 20만 건 가운데 무작위 2만 건(시드 0)의 별점 · 본문 |
| 리뷰 LLM 라벨(과정 결과, Qwen3-4B) | 2만 건의 리뷰 전체 극성, 무작위 순서의 앞 1,000건의 속성 라벨 |
| 공정거래위원회 소비자 민원(data.go.kr 15098314)과 민원 사건제목의 명사 파일(과정 결과) | 민원 표본 10만 건(시드 0). 앞 8만 건으로 상담이유 분류기를 학습하고 뒤 2만 건으로 평가 |

**과정 결과**는 이 교육 과정이 같은 표본 · 같은 프롬프트로 미리 만들어 둔 파일입니다. 이 실습은 모델을 호출하지 않고 저장된 라벨을 읽습니다.

- **리뷰 전체 극성**(LLM 라벨): 리뷰 한 건 전체가 긍정인지 부정인지입니다. 2만 건 모두에 있고, **별점 극성**(별점 1·2점은 부정, 4·5점은 긍정)과 비교할 수 있습니다.
- **불만 있음 · 없음**(LLM 라벨): 속성 라벨 가운데 근거가 확인된 부정 라벨이 하나라도 있으면 「있음」입니다. 속성 라벨이 있는 1,000건에만 있습니다.

별점은 리뷰 전체의 극성이라, 별점이 높은 리뷰에도 배송 불만이 있을 수 있습니다. 그래서 불만 있음 · 없음은 별점이 아니라 골드 표본으로 확인합니다.

아래 「자료 준비」 셀은 읽고 실행만 합니다.

In [ ]:
# 자료 준비: 읽고 실행만 합니다
# 리뷰 원자료 · 리뷰 LLM 라벨 · 민원 원자료 · 민원 사건제목의 명사 파일을 읽고, 단계 5 의 상담이유 분류기를 학습합니다. 모델은 호출하지 않습니다
import os, re, sys, json, math, time, hashlib, subprocess, urllib.request
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RAW_FILES = {
    "reviews": ("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt",
                "naver_shopping.txt",
                "dc4d1aca0a148671cbe80bcb81962eee297370acab42be93c1617ce9336479c0",
                "https://github.com/bab2min/corpus/tree/master/sentiment"),
    "complaints": ("https://www.data.go.kr/cmm/cmm/fileDownload.do?"
                   "atchFileId=FILE_000000002610054&fileDetailSn=1&insertDataPrcus=N",
                   "kftc_complaints_1.csv",
                   "50c098fea2ec1d05191d07c45cd0f75b4d0ac3b5e5ac36910fe40f2aab699ebe",
                   "https://www.data.go.kr/data/15098314/fileData.do"),
}
DATA_URL = "https://github.com/studio-js/korean-cx-text/raw/main/data"
MIRROR_URL = "https://github.com/studio-js/korean-cx-text/raw/main/raw"
SUBDIRS = ("", "cache", "raw", "processed")


def data_dir():
    return os.environ.get("NLP_DATA_DIR", "nlp_data")


def find_local(name):
    for sub in SUBDIRS:
        path = os.path.join(data_dir(), sub, name)
        if os.path.isfile(path):
            return path
    return None


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _download(url, path, retries=3):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    for i in range(retries):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            return True
        except Exception as e:
            print(f"받기 실패 {i + 1}/{retries}: {e}")
            time.sleep(2 * (i + 1))
    return False


def _fetch(url, name):
    path = find_local(name)
    if path is None:
        path = os.path.join(data_dir(), "cache", name)
        if not _download(url, path):
            return None
        os.replace(path + ".part", path)
    return path


def fetch_raw(name):
    url, fname, sha, page = RAW_FILES[name]
    path = find_local(fname)
    if path is None:
        path = _fetch(f"{MIRROR_URL}/{fname}", fname)
        if path is not None and sha256_of(path) != sha:
            os.remove(path)
            path = None
    if path is None:
        path = _fetch(url, fname)
    if path is None:
        raise RuntimeError(f"원 출처에서 파일을 받지 못했습니다. 데이터 페이지: {page}")
    if sha256_of(path) != sha:
        print(f"원 출처 파일이 바뀌었습니다. 결과 수가 본문의 수와 다를 수 있습니다(데이터 페이지: {page})")
    return path


def fetch_precomputed(name):
    path = _fetch(f"{DATA_URL}/{name}", name)
    if path is None:
        raise RuntimeError(f"과정 결과 파일 {name} 을 받지 못했습니다.")
    return path


def rnd(x, digits):
    x = float(x)
    if not math.isfinite(x):
        return x
    return float(Decimal(f"{x:.12g}").quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


def signed(x, digits):
    if round(x, digits) == 0:
        return f"{0:.{digits}f}"
    return f"{x:+.{digits}f}".replace("-", "−")


def fmt(x, digits):
    return f"{x:.{digits}f}".replace("-", "−")


PHONE_RE = re.compile(r"(?<!\d)(?:01[016789]|0\d{1,2})[-.)\s]{0,2}\d{3,4}[-.\s]\d{4}(?!\d)")


def llm_neg_of(status, labels):
    if status == "기타":
        return 0
    return int(any(x["polarity"] == "부정" and x.get("evidence_ok") for x in labels))


def llm_summary_of(status, labels):
    if status == "기타":
        return "라벨 실패"
    text = ", ".join(f"{x['aspect']}·{x['polarity']}" + ("" if x.get("evidence_ok") else "(보류)") for x in labels)
    return text or "속성 라벨 없음"


def parse_labels(text, n=30):
    if text is None or (isinstance(text, str) and not text.strip()):
        return None, f"아직 채우지 않았습니다. 단계 1 의 MY_NEG 에 번호 1～{n} 차례로 1 · 0 을 적은 뒤 다시 실행합니다"
    if not isinstance(text, str):
        return None, "고칠 것: MY_NEG 는 따옴표 안에 1 · 0 을 이어 적은 문자열입니다"
    s = re.sub(r"\s+", "", text)
    bad = sorted(set(ch for ch in s if ch not in "01"))
    if bad:
        return None, f"고칠 것: 1 · 0 밖의 글자({' · '.join(bad[:3])})가 있습니다. 영문 O 나 소문자 l 을 쓰지 않았는지 확인합니다"
    if len(s) < n:
        return None, f"{n}건 가운데 {len(s)}건만 적었습니다. 나머지도 적은 뒤 다시 실행합니다"
    if len(s) > n:
        return None, f"고칠 것: {len(s)}자를 적었습니다. 번호마다 한 글자씩 {n}자로 적습니다"
    return [int(ch) for ch in s], ""


def compare(mine, llm):
    n = len(mine)
    a = sum(m == 1 and q == 1 for m, q in zip(mine, llm))
    b = sum(m == 1 and q == 0 for m, q in zip(mine, llm))
    c = sum(m == 0 and q == 1 for m, q in zip(mine, llm))
    d = n - a - b - c
    me1, llm1 = (a + b) / n, (a + c) / n
    p_o = (a + d) / n
    p_e = me1 * llm1 + (1 - me1) * (1 - llm1)
    kappa = (p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")
    return {"a": a, "b": b, "c": c, "d": d, "me1": me1, "llm1": llm1, "p_o": p_o, "p_e": p_e, "kappa": kappa}


_t0 = time.time()
_rows = [ln.split("\t", 1) for ln in open(fetch_raw("reviews"), encoding="utf-8").read().split("\n") if ln]
REVIEWS = pd.DataFrame(_rows, columns=["별점", "본문"])
REVIEWS.insert(0, "row_id", range(len(REVIEWS)))
REVIEWS["별점"] = REVIEWS["별점"].astype(int)
SAMPLE = REVIEWS.sample(n=20000, random_state=0).reset_index(drop=True)
SAMPLE.insert(1, "rank", range(len(SAMPLE)))
_lab = pd.read_parquet(fetch_precomputed("llm_review_labels.parquet")).drop(columns=["rank"], errors="ignore")
_m = SAMPLE.merge(_lab, on="row_id", how="left")
LLM20 = (_m["overall"] == "부정").to_numpy().astype(int)
STAR20 = (_m["별점"] <= 2).to_numpy().astype(int)
LABELS_1000 = _m[_m["aspects_done"].eq(True)].sort_values("rank").reset_index(drop=True)
LABELS_1000["labels"] = [json.loads(s) if isinstance(s, str) else [] for s in LABELS_1000["aspects_json"]]
LABELS_1000["llm_neg"] = [llm_neg_of(st, lb) for st, lb in zip(LABELS_1000["status"], LABELS_1000["labels"])]
GOLD30 = LABELS_1000[LABELS_1000["rank"].between(500, 529)].reset_index(drop=True)
GOLD30.insert(0, "번호", range(1, len(GOLD30) + 1))
GOLD30["본문"] = GOLD30["본문"].map(lambda t: PHONE_RE.sub("[전화번호]", t))
GOLD30["llm_labels"] = [llm_summary_of(st, lb) for st, lb in zip(GOLD30["status"], GOLD30["labels"])]
MODEL_INFO = f"{_lab['model'].iloc[0].split('/')[-1]}(버전 {_lab['model_revision'].iloc[0][:8]})"
VERSIONS = (_lab["prompt_version_overall"].iloc[0], LABELS_1000["prompt_version_aspects"].iloc[0])

_len = SAMPLE["본문"].str.len().to_numpy()
CUT = int(np.median(_len))
GROUPS = []
for _name, _mask in ((f"짧은 리뷰({CUT}자 이하)", _len <= CUT), (f"긴 리뷰({CUT}자 초과)", _len > CUT)):
    _a, _s = LLM20[_mask], STAR20[_mask]
    GROUPS.append({"name": _name, "n": int(_mask.sum()), "recall": round(_a[_s == 1].mean(), 3),
                   "fpr": round(_a[_s == 0].mean(), 3), "star": round(100 * _s.mean(), 1), "llm": round(100 * _a.mean(), 1),
                   "miss": round(100 * ((_s == 1) & (_a == 0)).mean(), 1), "fp": round(100 * ((_s == 0) & (_a == 1)).mean(), 1)})

TEST_INPUT_RE = re.compile(
    r"(?i)^\s*(?:test|테스트)\s*(?:입니다)?\s*\.?\s*$"
    r"|IN-TEST"
    r"|(?:인바운드|아웃바운드|녹취|수화기).*테스트|테스트.*(?:인바운드|아웃바운드|녹취|수화기)"
    r"|^\s*자동차\s*(?:상담\s*)?테스트"
    r"|^\s*\d{8}\s*[-_ ].*테스트"
    r"|테스트\s*데이터|상담\s*(?:전화\s*)?테스트|전화\s*테스트|헬프데스크\s*시험|^\s*일반\s*테스트")
EMPTY_TITLE_RE = re.compile(r"^[^가-힣A-Za-z]*$")
_c = pd.read_csv(fetch_raw("complaints"), encoding="cp949", dtype=str)
_c = _c.rename(columns=lambda c: c.split("(")[0].strip())
_c.insert(0, "row_id", range(len(_c)))
_title = _c["사건제목"].fillna("").str.strip()
_test = _title.str.contains(TEST_INPUT_RE)
_c = _c[~_test & ~(_title.str.match(EMPTY_TITLE_RE) & ~_test)].reset_index(drop=True)
C100K = _c.sample(n=100000, random_state=0)[["row_id", "상담이유명"]].reset_index(drop=True)
try:
    C100K = C100K.merge(pd.read_parquet(fetch_precomputed("complaints_nouns.parquet")), on="row_id", how="left")
except Exception as e:
    print(f"민원 사건제목의 명사 파일을 받지 못해 Kiwi 로 10만 건의 명사를 다시 구합니다(Colab CPU 1분 안팎): {e}")
    try:
        from kiwipiepy import Kiwi
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy==0.23.1"], check=True)
        from kiwipiepy import Kiwi
    os.environ.setdefault("KIWI_ARCH_TYPE", "balanced")
    _titles = (_c.set_index("row_id").loc[C100K["row_id"], "사건제목"].fillna("").str.strip()
               .map(lambda t: PHONE_RE.sub("[전화번호]", t)))
    C100K["명사"] = [[t.form for t in toks if t.tag in ("NNG", "NNP")]
                     for toks in Kiwi(num_workers=2).tokenize(list(_titles))]
_docs = [" ".join(ns) for ns in C100K["명사"]]
_y = C100K["상담이유명"].to_numpy()
_vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, min_df=2)
_clf = LogisticRegression(max_iter=2000).fit(_vec.fit_transform(_docs[:80000]), _y[:80000])
_pred = _clf.predict(_vec.transform(_docs[80000:]))
_ytest = _y[80000:]
_top = pd.Series(_y[:80000]).value_counts().idxmax()
CLF = {"acc": (_pred == _ytest).mean(), "base": (_ytest == _top).mean(), "top": _top, "n_classes": len(set(_y)),
       "rare": [{"name": r, "n": int((_ytest == r).sum()), "hit": int(((_ytest == r) & (_pred == r)).sum())}
                for r in ["부당채권추심", "무능력자계약"]],
       "by_class": [{"name": r, "n": int((_ytest == r).sum()), "hit": int(((_ytest == r) & (_pred == r)).sum())}
                    for r in sorted(set(_ytest))]}

_k = int(LABELS_1000["llm_neg"].sum())
print(f"준비 끝: 리뷰 표본 {len(SAMPLE):,}건 · 불만 있음 · 없음 LLM 라벨 {len(LABELS_1000):,}건 · 골드 표본 {len(GOLD30)}건 · 평가 민원 {len(_ytest):,}건")
print(f"리뷰 {len(REVIEWS):,}건 → 리뷰 표본 {len(SAMPLE):,}건(시드 0) · 리뷰 전체 극성과 별점 극성 모두 {len(SAMPLE):,}건")
print(f"불만 있음 · 없음 LLM 라벨: 무작위 순서의 앞 {len(LABELS_1000):,}건 · 「있음」 {_k}건({100 * _k / len(LABELS_1000):.1f}%)"
      f" · 라벨 실패 {int((LABELS_1000['status'] == '기타').sum())}건")
print(f"골드 표본: 무작위 순서의 501～530번째 리뷰 {len(GOLD30)}건 · LLM 라벨은 단계 1 의 확인 셀에서 보여 줍니다")
print(f"모델 {MODEL_INFO} · 프롬프트 버전 {VERSIONS[0]}(리뷰 전체 극성) · {VERSIONS[1]}(속성 라벨)")
print(f"리뷰 길이 두 집단: 본문 {CUT}자 이하 {GROUPS[0]['n']:,}건 · {CUT}자 초과 {GROUPS[1]['n']:,}건")
print(f"민원: 표본 {len(C100K):,}건(시드 0) · 앞 80,000건 학습 · 뒤 20,000건 평가 · 상담이유 {CLF['n_classes']}종")
print(f"준비에 걸린 시간 {time.time() - _t0:.0f}초")

준비 끝: 리뷰 표본 20,000건 · 불만 있음 · 없음 LLM 라벨 1,000건 · 골드 표본 30건 · 평가 민원 20,000건
리뷰 200,000건 → 리뷰 표본 20,000건(시드 0) · 리뷰 전체 극성과 별점 극성 모두 20,000건
불만 있음 · 없음 LLM 라벨: 무작위 순서의 앞 1,000건 · 「있음」 523건(52.3%) · 라벨 실패 19건
골드 표본: 무작위 순서의 501～530번째 리뷰 30건 · LLM 라벨은 단계 1 의 확인 셀에서 보여 줍니다
모델 Qwen3-4B-Instruct-2507(버전 cdbee75f) · 프롬프트 버전 overall-v1(리뷰 전체 극성) · aspect-v1(속성 라벨)
리뷰 길이 두 집단: 본문 29자 이하 10,151건 · 29자 초과 9,849건
민원: 표본 100,000건(시드 0) · 앞 80,000건 학습 · 뒤 20,000건 평가 · 상담이유 16종
준비에 걸린 시간 8초

In [ ]:
# 자료 준비 · 그림: 리뷰 1,000건의 불만 있음 · 없음 LLM 라벨(라벨 실패는 따로)
yes = int(LABELS_1000["llm_neg"].sum())
fail = int((LABELS_1000["status"] == "기타").sum())
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(["불만 있음", "불만 없음", "라벨 실패"], [yes, len(LABELS_1000) - yes - fail, fail], color=["#2251FF", "gray", "#B3B3B3"])
ax.invert_yaxis()
ax.set_xlabel("리뷰 수")
ax.set_title(f"리뷰 1,000건 가운데 LLM 라벨이 「있음」인 리뷰는 {yes}건입니다")
plt.show()

리뷰 1,000건 가운데 LLM 라벨이 「있음」인 리뷰는 523건(52.3%)입니다. 라벨 실패 19건은 분모에 남고 「있음」으로 집계하지 않습니다. 이 52.3% 는 LLM 라벨만으로 구한 값이라, 라벨이 틀린 만큼 참값과 다를 수 있습니다.

골드 표본 30건은 이 1,000건 안의 무작위 리뷰입니다. 30건의 LLM 라벨도 1,000건과 같은 모델 · 같은 프롬프트 버전으로 만들었습니다. 그래서 골드 표본에서 구한 보정값을 1,000건에 더할 수 있습니다.

## 단계 1 골드 표본 30건 라벨링하기

**골드 표본**은 무작위로 고른 리뷰를 LLM 라벨을 보지 않고 사람이 직접 라벨링해 참값으로 삼는 표본입니다. 여기서는 리뷰 30건을 직접 라벨링합니다. 30건은 무작위 순서에서 이어진 리뷰라 무작위로 고른 30건과 같습니다.

**라벨 기준**: 코드북 v1 의 6개 속성 가운데 하나라도 부정으로 평가한 구절이 있으면 1, 없으면 0 입니다. 「택배는 진짜 느리네요」는 1, 「너무 좋아요. 진작 살걸요ㅎ」는 0 입니다.

| 속성 | 정의 | 포함 기준 | 제외 기준 |
|---|---|---|---|
| 배송 | 주문한 물건이 도착하기까지의 과정 | 속도·지연, 택배, 포장·박스 상태, 오배송 | 제품 자체의 흠집·불량(→ 품질) |
| 품질 | 제품 자체의 상태와 성능 | 불량, 마감, 내구성, 맛·신선도, 성능, 디자인 | 크기·치수(→ 사이즈), 포장 상태(→ 배송) |
| 사이즈 | 크기·길이·치수가 맞는지 | 크기, 길이, 핏, 양 | 무게감·두께 같은 재질 느낌(→ 품질) |
| 가격 | 값과 가성비 | 싸다·비싸다, 가성비, 할인 | 환불 금액(→ 교환/환불) |
| 교환/환불 | 교환·반품·환불·취소 절차 | 교환·반품, 환불, 주문 취소 | 처음 배송 지연(→ 배송) |
| 기타 | 위 5개 속성에 속하지 않는 구체적 평가 | 판매자 응대, 사은품 등 | 속성이 드러나지 않는 감탄(예: 좋아요)은 라벨 없음 |

- LLM 라벨은 근거 텍스트가 원문에서 확인된 부정 속성 라벨이 하나라도 있으면 1 입니다. LLM 라벨링 때의 라벨 실패(재시도 뒤에도 형식을 어긴 출력)와 보류한 라벨(근거 텍스트가 원문에 없는 라벨)로는 1 이 되지 않습니다.
- 이 실습에서는 내 라벨을 참값으로 씁니다. 사람 라벨도 틀릴 수 있어서, 실제로는 두 사람이 따로 라벨링하고 두 사람의 κ 를 구합니다.
- 판단이 갈리는 애매한 리뷰는 어느 쪽으로 정했는지 기억해 둡니다. 이런 리뷰는 코드북의 포함 · 제외 기준에 적어 다음 버전에 반영합니다.

In [ ]:
# 단계 1: 골드 표본 30건의 번호와 본문(LLM 라벨은 확인 셀에서 보여 줍니다)
print("골드 표본 30건. 코드북 v1 의 속성 가운데 하나라도 부정으로 평가한 구절이 있으면 1, 없으면 0")
for k, text in zip(GOLD30["번호"], GOLD30["본문"]):
    print(f"{k:>2}. {text}")

골드 표본 30건. 코드북 v1 의 속성 가운데 하나라도 부정으로 평가한 구절이 있으면 1, 없으면 0
 1. 가벼운물건위주로써야할듯하네요
 2. 저희아이한테는 잘안맞는거같아요
 3. 재구매 재구매 제품인데 처움고다 맛 질이 안좋네요 같이 온 연어 넘 좋구 맛 있어요
 4. 너무너무마음에듭니다.잘쓰겠습니다.
 5. 크리스탈라이트 살거 그랬나봐요
 6. 배송이직구치고 빠르네요. 국내보다 저렴한것같아요.
 7. 좋아요 하나하ㅏ사는것보다 가격도 싸고 품질도 앵간쓰
 8. 확실히 가격이 더나가는게 이뻐요~
 9. 제껀 진공이 풀려서 도착했지만 맛은 아주 맛있어요!!! 또 살꼬임
10. 완전맘에들어요~^^
11. 사진과 다르네요
12. 꼭지 원형둘레 사이를 청소하기 힘듬
13. 아직 사용안해봐서 바람이 빠지는지는 모르겠구요 제품에 페인트뭍은거처럼 하얀색 물방울모양이 여기저기 뭍어있네요? 물건을 어떻게 관리하길래 이렇게 하시는거죠? 전화할라다가 시간이 늦어서 그냥놔둡니다
14. 뚜껑이 잘안열리고 잘안닫혀요 반품하기 귀찮아서 그냥 사용하고 있습니다.
15. 가격정도의 가치 그이상도아닙니다
16. 좋아요ㅎ 다른데 시켰는데 배송이 하도 늦어서 여기서 다시 시켰는데 바로 왔어요ㅎ 배송 진짜 빠르네요ㅎ 케이스도 마음이 들어요ㅎ
17. 아주 좋아요 피*보다 더 좋은듯!!
18. 깔끔해서 이뻐요. 여유로운 사이즈로 사세용
19. 1주일만에 뜯어져버렸네요. 만원 더들여서 그냥 메이커 사주는게..
20. 좋습니다. 사용도 간편하고,커피도 신선하게 맛있게추출되서 자주자주 사용할듯합니다.AS같은것만 잘된다면 평생사용할것같아요
21. 항상맛있는 광천김 대량구매했어요
22. 재구매 설연휴 이후에 바로 시켜서 그런가 배송은 펼소보다 좀 늦었지만... 맛은 한결같이 맛있어요. 받자 마자 냉장고에 넣고 한포기씩 실온에 뒀다가 먹어요 다먹고 다시 주문할겁니다 ㅎㅎ
23. 요녀석덕에 욕실이 깔끔❤정말너무 실용적이구 위생적이예요ㅋㅋ사이즈도딱좋아요ㅋ큰칸엔 샴푸캡도 딱ㅋㅋ
24. 또 살 의향있어요 배송도 빠

**직접 해 보기: 30건의 라벨 적기**

위 셀이 출력한 30건을 읽고 `MY_NEG` 에 번호 1～30 차례로 1 · 0 을 적습니다. 30자를 이어 적고, 5자마다 띄어 적어도 됩니다.

- 적는 방법: `MY_NEG = "○○○○○ ○○○○○ ○○○○○ ○○○○○ ○○○○○ ○○○○○"`(○ 하나가 리뷰 하나이고 1 이나 0)
- 라벨은 AI 에 맡기지 않고 직접 정합니다. 내 라벨이 이 실습의 참값입니다.
- 두 라벨이 다른 리뷰는 **나만 있음**(내 라벨만 1)과 **LLM 만 있음**(LLM 라벨만 1)으로 나눕니다.

두 라벨이 다른 리뷰마다 LLM 출력이 나오므로 어느 구절에서 판단이 갈렸는지 비교할 수 있습니다.

In [ ]:
# 단계 1 · 직접 해 보기: 번호 1～30 차례로 불만 있음 1 · 없음 0 (30자, 5자마다 띄어 적어도 됩니다)
MY_NEG = ""

In [ ]:
# 단계 1 · 확인 셀: MY_NEG 를 LLM 라벨과 비교합니다(이 실습에서는 내 라벨이 참값)
def show_diff(mine, k):
    row = GOLD30.iloc[k - 1]
    side = "나만 있음" if mine[k - 1] == 1 else "LLM 만 있음"
    text = row["본문"] if len(row["본문"]) <= 40 else row["본문"][:40] + "…"
    print(f"  {k}번 · {side} · LLM 출력: {row['llm_labels']} · {text}")


def check_gold(text):
    mine, msg = parse_labels(text)
    if mine is None:
        print(msg)
        return
    r = compare(mine, GOLD30["llm_neg"].tolist())
    print(f"일치율 {r['p_o']:.3f} · 코헨 카파 κ {fmt(r['kappa'], 3)} · 두 라벨이 다른 리뷰 {r['b'] + r['c']}건"
          f"(LLM 만 있음 {r['c']}건 · 나만 있음 {r['b']}건)")
    print(f"나 있음: LLM 있음 {r['a']}건 · LLM 없음 {r['b']}건")
    print(f"나 없음: LLM 있음 {r['c']}건 · LLM 없음 {r['d']}건")
    print(f"우연 일치 {r['p_e']:.3f} = 나 {r['me1']:.3f} × LLM {r['llm1']:.3f}(있음끼리)"
          f" + 나 {1 - r['me1']:.3f} × LLM {1 - r['llm1']:.3f}(없음끼리)")
    diff = [k for k in range(1, 31) if mine[k - 1] != GOLD30["llm_neg"].iloc[k - 1]]
    if diff:
        print("두 라벨이 다른 리뷰")
    for k in diff[:6]:
        show_diff(mine, k)
    if len(diff) > 6:
        print(f"  … 외 {len(diff) - 6}건")


check_gold(globals().get("MY_NEG", ""))

아직 채우지 않았습니다. 단계 1 의 MY_NEG 에 번호 1～30 차례로 1 · 0 을 적은 뒤 다시 실행합니다

In [ ]:
# 단계 1 · 그림: 골드 표본 30건의 일치표(MY_NEG 를 채운 뒤에 그려집니다)
mine, msg = parse_labels(globals().get("MY_NEG", ""))
if mine is None:
    print("단계 1 의 MY_NEG 를 30자로 채우면 일치표 그림이 그려집니다")
else:
    r = compare(mine, GOLD30["llm_neg"].tolist())
    grid = np.array([[r["a"], r["b"]], [r["c"], r["d"]]])
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    ax.imshow(grid, cmap="Blues", vmin=0, vmax=30)
    for (i, j), v in np.ndenumerate(grid):
        ax.text(j, i, f"{v}건", ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1], ["LLM 있음", "LLM 없음"])
    ax.set_yticks([0, 1], ["나 있음", "나 없음"])
    ax.set_title(f"골드 표본 30건의 일치표 · 일치율 {r['p_o']:.3f}")
    plt.show()

단계 1 의 MY_NEG 를 30자로 채우면 일치표 그림이 그려집니다

<details class="lms-extra"><summary>자세히 — 예시 답</summary>

`MY_NEG = "11100 00010 11110 00010 01000 01111"`

- 둘 다 있음 11건 · 나만 있음 3건 · LLM 만 있음 2건 · 둘 다 없음 14건입니다.
- 일치율은 25 ÷ 30 = 0.833, 우연 일치는 0.504, κ 는 0.664 입니다.
- 번호 13 은 LLM 도 품질 부정을 냈지만, 근거 텍스트가 원문에 없는 보류 라벨이라 1 이 되지 않았습니다. 번호 30 은 LLM 출력이 라벨 실패라 0 입니다.
- 번호 7 「품질도 앵간쓰」와 번호 15 「가격정도의 가치 그이상도아닙니다」는 칭찬도 불평도 아닌 평가입니다. 두 리뷰는 이 예시가 0, LLM 이 1 로 판단한 애매한 리뷰입니다.
- 번호 1 「가벼운물건위주로써야할듯하네요」는 무거운 물건에 약하다는 품질 부정으로 판단했습니다. LLM 은 라벨을 내지 않았습니다.

κ 0.664 는 이론에서 본 Landis · Koch 의 구분으로 「상당한 일치」입니다. 다만 30건으로 구한 일치율은 표본마다 크게 달라질 수 있습니다.

</details>

> **[문제] 「있음」이 드문 두 라벨의 κ**
>
> 예를 들어 리뷰 100건을 사람과 LLM 이 따로 라벨링했습니다. 둘 다 「있음」을 10건씩 골랐고 일치율은 0.90 입니다. κ 는 얼마일까요?
>
> 1. 약 0.90: 일치율과 κ 는 같은 값이다
> 2. 약 0.80: 우연 일치는 언제나 0.5 다
> 3. 약 0.44: 우연 일치가 0.82 로 크다

## 단계 2 κ 가 높으면 비율도 맞는가

κ 가 높으면 LLM 라벨로 구한 비율도 맞을까요? 꼭 그렇지 않습니다. 비율은 LLM 만 있음과 참값만 있음의 건수 차이에 달려 있습니다.

**편향(bias)** 은 LLM 라벨 비율에서 참값 비율을 뺀 값입니다.

```text
편향 = LLM 라벨 비율 − 참값 비율 = (LLM 만 있음 − 참값만 있음) ÷ 전체 건수
```

골드 표본에서는 내 라벨이 참값이라 참값만 있음이 단계 1 의 나만 있음입니다. 리뷰 2만 건에서는 별점 극성이 참값이라 별점만 부정입니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/5/5d/Reliability_and_validity.svg" alt="과녁 네 개. 왼쪽 위는 점이 넓게 퍼지고 가운데에서 비켜남, 오른쪽 위는 넓게 퍼졌지만 가운데를 에워쌈, 왼쪽 아래는 좁게 뭉쳤지만 가운데에서 비켜남, 오른쪽 아래는 가운데에 좁게 뭉침" width="520" height="557" />
<p align="center"><em>▲ 과녁 네 개에서 점 하나는 표본 한 번으로 구한 값이고, 과녁 가운데는 참값 비율이다. 점이 좁게 모일수록 표본마다 값이 덜 달라지고, 가운데에서 떨어진 거리가 편향이다. 왼쪽 아래(Reliable, Not Valid)는 값이 일정하지만 가운데를 맞히지 못한 경우다 (출처: Wikimedia Commons, Reliability and validity.svg, Nevit Dilmen, CC BY-SA 3.0)</em></p>

아래 셀은 골드 표본 30건과 리뷰 2만 건을 같은 방법으로 비교합니다. 리뷰 2만 건은 LLM 의 리뷰 전체 극성을 별점 극성과 비교합니다.

In [ ]:
# 단계 2: 골드 표본 30건(참값은 내 라벨)과 리뷰 2만 건(참값은 별점 극성)의 두 방향 건수
mine, msg = parse_labels(globals().get("MY_NEG", ""))
if mine is None:
    print("골드 표본 30건: 단계 1 의 MY_NEG 를 30자로 채우면 여기에 출력됩니다")
else:
    g = compare(mine, GOLD30["llm_neg"].tolist())
    print(f"골드 표본 30건: 불만 비율 LLM 라벨 {g['llm1']:.3f} · 내 라벨 {g['me1']:.3f} · 편향 {signed(g['llm1'] - g['me1'], 3)}")
    print(f"  LLM 만 있음 {g['c']}건 · 나만 있음 {g['b']}건 · 일치율 {g['p_o']:.3f} · κ {fmt(g['kappa'], 3)}")
s = compare(STAR20.tolist(), LLM20.tolist())
llm_pct, star_pct = rnd(100 * LLM20.mean(), 2), rnd(100 * STAR20.mean(), 2)
print()
print("표 1 · 리뷰 2만 건의 부정 비율: LLM 라벨과 별점 (%, 건)")
print(f"LLM 라벨 부정 비율 {llm_pct:.2f}% · 별점 부정 비율 {star_pct:.2f}% · 편향 {signed(llm_pct - star_pct, 2)}%p")
print(f"LLM 부정: 별점 부정 {s['a']:,}건 · 별점 긍정 {s['c']:,}건")
print(f"LLM 긍정: 별점 부정 {s['b']:,}건 · 별점 긍정 {s['d']:,}건")
print(f"두 방향: LLM 라벨만 부정 {s['c']:,}건 − 별점만 부정 {s['b']:,}건 = {s['c'] - s['b']}건")
print(f"일치율 {s['p_o']:.3f} · 우연 일치 {s['p_e']:.3f} · κ {s['kappa']:.2f}")

골드 표본 30건: 단계 1 의 MY_NEG 를 30자로 채우면 여기에 출력됩니다

표 1 · 리뷰 2만 건의 부정 비율: LLM 라벨과 별점 (%, 건)
LLM 라벨 부정 비율 51.39% · 별점 부정 비율 49.74% · 편향 +1.65%p
LLM 부정: 별점 부정 9,311건 · 별점 긍정 967건
LLM 긍정: 별점 부정 636건 · 별점 긍정 9,086건
두 방향: LLM 라벨만 부정 967건 − 별점만 부정 636건 = 331건
일치율 0.920 · 우연 일치 0.500 · κ 0.84

In [ ]:
# 단계 2 · 그림: 리뷰 2만 건에서 두 라벨이 다른 리뷰의 두 방향
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.barh(["LLM 라벨만 부정", "별점만 부정"], [s["c"], s["b"]], color=["#2251FF", "gray"])
ax.invert_yaxis()
ax.set_xlabel("리뷰 수")
ax.set_title(f"κ {s['kappa']:.2f} 인데도 LLM 라벨만 부정인 리뷰가 {s['c'] - s['b']}건 더 많습니다")
plt.show()

> **[문제] 리뷰 2만 건의 부정 비율을 구하는 방법**
>
> 「표 1 · 리뷰 2만 건의 부정 비율: LLM 라벨과 별점 (%, 건)」에서 리뷰 2만 건의 부정 비율로 어느 값을 쓸까요?
>
> 1. LLM 라벨 부정 비율: κ 가 0.8 을 넘어 두 라벨이 거의 같다
> 2. 별점 부정 비율: 2만 건 모두 별점이 있고 LLM 라벨은 편향되었다
> 3. 두 비율의 평균: 어느 쪽이 맞는지 알 수 없다

**직접 해 보기: 편향의 방향 · 리뷰 2만 건의 부정 비율을 구하는 방법**

- `MY_MORE`: 단계 1 에서 두 라벨이 다른 리뷰 가운데 많은 쪽입니다. LLM 만 있음이 많으면 "LLM", 나만 있음이 많으면 "나", 두 건수가 같으면 "같음" 입니다.
- `MY_USE`: 리뷰 2만 건의 부정 비율을 구하는 방법입니다. 별점 부정 비율이면 "별점", LLM 라벨 부정 비율이면 "LLM", 두 비율의 평균이면 "평균" 입니다.

In [ ]:
# 단계 2 · 직접 해 보기: 두 방향 가운데 많은 쪽("LLM" · "나" · "같음") · 리뷰 2만 건의 부정 비율을 구하는 방법("별점" · "LLM" · "평균")
MY_MORE = None
MY_USE = None

In [ ]:
# 단계 2 · 확인 셀: MY_MORE 는 단계 1 의 MY_NEG 로 다시 계산해 비교하고, MY_USE 는 표 1 로 판단합니다
def check_bias(more, use):
    if more is None and use is None:
        print("아직 채우지 않았습니다. 위 셀의 MY_MORE · MY_USE 를 채운 뒤 다시 실행합니다")
        return
    s = compare(STAR20.tolist(), LLM20.tolist())
    gap = rnd(100 * LLM20.mean(), 2) - rnd(100 * STAR20.mean(), 2)
    why = {"별점": "2만 건 모두 별점이 있어 따로 추정할 필요가 없습니다",
           "LLM": f"κ {s['kappa']:.2f} 여도 LLM 라벨만 부정인 리뷰가 {s['c'] - s['b']}건 더 많아 편향이 {gap:.2f}%p 입니다",
           "평균": f"별점이 2만 건 모두에 있는데 평균을 내면 편향 {gap:.2f}%p 의 절반이 남습니다"}
    if more is None or use is None:
        print(f"{'MY_MORE' if more is None else 'MY_USE'} 는 아직 채우지 않았습니다. 채운 변수만 확인합니다")
    if more is not None and (not isinstance(more, str) or more not in ("LLM", "나", "같음")):
        print('고칠 것: MY_MORE 는 "LLM" · "나" · "같음" 가운데 하나로 적습니다')
    elif more is not None:
        mine, msg = parse_labels(globals().get("MY_NEG", ""))
        if mine is None:
            print("MY_MORE: 단계 1 의 MY_NEG 를 30자로 채운 뒤 다시 실행하면 확인합니다")
        else:
            g = compare(mine, GOLD30["llm_neg"].tolist())
            right = "LLM" if g["c"] > g["b"] else "나" if g["b"] > g["c"] else "같음"
            tie = '. 두 건수가 같으면 "같음" 을 적습니다' if right == "같음" and more != right else ""
            print(f"MY_MORE: {'맞았습니다' if more == right else '아직 다릅니다'}. 골드 표본 30건에서 LLM 만 있음 {g['c']}건 · 나만 있음 {g['b']}건입니다{tie}")
    if use is not None and (not isinstance(use, str) or use not in why):
        print('고칠 것: MY_USE 는 "별점" · "LLM" · "평균" 가운데 하나로 적습니다')
    elif use is not None:
        print(f"MY_USE: {'맞았습니다' if use == '별점' else '아직 다릅니다'}. {why[use]}")


check_bias(globals().get("MY_MORE"), globals().get("MY_USE"))

아직 채우지 않았습니다. 위 셀의 MY_MORE · MY_USE 를 채운 뒤 다시 실행합니다

<details class="lms-extra"><summary>자세히 — 예시 답</summary>

`MY_MORE = "나"` · `MY_USE = "별점"`

- 나만 있음 3건이 LLM 만 있음 2건보다 많아 "나" 입니다. `MY_MORE` 의 답은 내 라벨에 따라 달라집니다.
- 이 30건에서는 LLM 라벨의 불만 비율이 내 라벨보다 0.033 낮습니다. 두 라벨이 다른 리뷰가 5건뿐이라 편향의 방향을 단정하기 어렵습니다.
- 리뷰 2만 건은 κ 0.84 로 「거의 완전한 일치」입니다. 그래도 LLM 라벨만 부정인 리뷰가 별점만 부정인 리뷰보다 331건 많아, LLM 라벨 부정 비율이 1.65%p 높습니다.
- 2만 건 모두 별점이 있으므로 리뷰 전체 부정 비율은 별점 49.74% 를 씁니다. 두 비율의 평균은 편향의 절반이 남은 값입니다.

</details>

## 단계 3 보정한 비율

참값이 골드 표본 30건에만 있을 때, 리뷰 1,000건의 불만 비율은 무엇을 써야 할까요? LLM 라벨로 구한 비율을 골드 표본에서 본 편향만큼 고친 보정한 비율을 씁니다. 그 값이 얼마나 불확실한지는 95% 구간으로 나타냅니다.

내 라벨 30건의 비율은 그대로 쓰지 않습니다. 30건만으로 구한 비율은 95% 구간 길이가 최대 2 × 1.96 × √(0.25 ÷ 30) ≈ 35.8%p 로 넓습니다. 1,000건의 LLM 라벨을 쓰고 골드 표본으로는 편향만 고치면, 두 라벨이 같은 리뷰가 많을수록 구간이 짧아집니다.

```text
보정값      = 골드 표본 30건의 (내 라벨 − LLM 라벨) 평균
보정한 비율 = 리뷰 1,000건의 LLM 라벨 불만 비율 + 보정값
```

- 리뷰마다 차이는 나만 있음이면 +1, LLM 만 있음이면 −1, 두 라벨이 같으면 0 입니다. 보정값은 (나만 있음 − LLM 만 있음) ÷ 30 과 같습니다.
- 95% 구간은 이론의 부트스트랩과 같은 방법으로 구합니다. 30건의 차이를 복원 추출로 다시 골라 보정한 비율을 2,000번 구하고, 2.5 · 97.5 백분위수를 씁니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/17fig-boot-ci.png" alt="종 모양 곡선 아래 가운데 95% 가 연두색으로 칠해져 있고, 왼쪽 0.025 Quantile 과 오른쪽 0.975 Quantile 에 빨간 세로 점선이 있다. 두 점선 사이 화살표에 Plausible Interval for Actual Value of p, 바깥 두 꼬리에 Lower 2.5% 와 Upper 2.5% 라고 적혀 있다" width="560" height="346"/>
<p align="center"><em>▲ 부트스트랩 백분위수 95% 구간. 복원 추출을 되풀이해 구한 값들의 분포에서 아래 2.5% 와 위 2.5% 를 잘라 낸 가운데 95% 가 구간이다. 단계 3 · 4 에서는 보정한 비율 2,000개의 분포다 (출처: Wikimedia Commons, 17fig-boot-ci.png, ASpiegler, CC BY-SA 4.0)</em></p>

**직접 해 보기: 보정한 비율 예측하기**

아래 계산을 실행하기 전에, 보정한 비율이 1,000건의 LLM 라벨 불만 비율보다 어떨지 예측해 `MY_GUESS` 에 적습니다. "높다" · "낮다" · "같다" 가운데 하나입니다. 단계 2 에서 확인한 두 방향의 건수로 정합니다.

In [ ]:
# 단계 3 · 직접 해 보기(실행 전): 보정한 비율이 LLM 라벨 불만 비율보다 "높다" · "낮다" · "같다"
MY_GUESS = None

In [ ]:
# 단계 3: 리뷰 1,000건의 불만 비율을 골드 표본 30건으로 보정하고, 부트스트랩으로 95% 구간을 구합니다
P_LLM = LABELS_1000["llm_neg"].mean()                         # 1,000건의 LLM 라벨 불만 비율
mine, msg = parse_labels(globals().get("MY_NEG", ""))
PPI = None
if mine is None:
    print(f"리뷰 1,000건의 LLM 라벨 불만 비율 {100 * P_LLM:.1f}% · 보정하려면 단계 1 의 MY_NEG 를 30자로 채웁니다")
else:
    diff = np.array(mine) - GOLD30["llm_neg"].to_numpy()      # 리뷰마다 (내 라벨 − LLM 라벨): −1 · 0 · +1
    rng = np.random.default_rng(0)
    boots = [P_LLM + diff[rng.integers(0, 30, 30)].mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    raw_vals = {"est": 100 * (P_LLM + diff.mean()), "lo": 100 * lo, "hi": 100 * hi}
    PPI = {k: min(max(v, 0.0), 100.0) for k, v in raw_vals.items()}   # 0～100% 밖이면 끝값으로 표시
    PPI.update(adj=100 * diff.mean(), plus=int((diff == 1).sum()), minus=int((diff == -1).sum()), boots=100 * np.array(boots),
               neg="".join(map(str, mine)))
    p30 = np.mean(mine)                                       # 내 라벨 30건만 쓴 비율(참값만 쓰는 방법)
    h30 = 1.96 * np.sqrt(p30 * (1 - p30) / 30)
    print("표 2 · 리뷰 1,000건의 불만 비율: 골드 표본 30건으로 보정한 비율")
    print(f"보정한 비율 {PPI['est']:.1f}%(95% 구간 {PPI['lo']:.1f}～{PPI['hi']:.1f}%) · 구간 길이 {PPI['hi'] - PPI['lo']:.1f}%p")
    print(f"LLM 라벨 불만 비율 {100 * P_LLM:.1f}%: 리뷰 1,000건 가운데 {int(LABELS_1000['llm_neg'].sum())}건")
    print(f"보정값 {signed(PPI['adj'], 1)}%p: (나만 있음 {PPI['plus']}건 − LLM 만 있음 {PPI['minus']}건) ÷ 30")
    op = "+" if PPI["adj"] >= 0 else "−"
    print(f"보정한 비율 = {100 * P_LLM:.1f}% {op} {abs(PPI['adj']):.1f}%p = {raw_vals['est']:.1f}%")
    print("95% 구간: 30건의 차이를 복원 추출로 2,000번 다시 고른 보정한 비율의 2.5 · 97.5 백분위수(시드 0)")
    print(f"내 라벨 30건만 쓴 비율 {100 * p30:.1f}%(95% 구간 {max(0.0, 100 * (p30 - h30)):.1f}～{min(100.0, 100 * (p30 + h30)):.1f}%)"
          f" · 구간 길이 {200 * h30:.1f}%p")
    if any(PPI[k] != raw_vals[k] for k in raw_vals):
        print("0～100% 밖으로 나간 값은 0% 나 100% 로 표시했습니다. 나만 있음과 LLM 만 있음의 차이가 크면 생깁니다")

리뷰 1,000건의 LLM 라벨 불만 비율 52.3% · 보정하려면 단계 1 의 MY_NEG 를 30자로 채웁니다

In [ ]:
# 단계 3 · 그림: 다시 고른 보정한 비율 2,000개(값마다 막대 하나)와 95% 구간
PPI = globals().get("PPI")
if PPI is None:
    print("단계 1 의 MY_NEG 를 채운 뒤 위 셀을 실행하면 그림이 그려집니다")
else:
    vals, counts = np.unique(np.round(PPI["boots"], 2), return_counts=True)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(vals, counts, width=2.5, color="#2251FF")
    ax.vlines([PPI["lo"], PPI["hi"]], 0, counts.max(), colors="#051C2C", linestyles="--", label="95% 구간의 두 끝")
    ax.set_ylim(0, counts.max() * 1.3)
    ax.set_xlabel("보정한 비율(%)")
    ax.set_ylabel("2,000번 가운데 횟수")
    ax.legend(frameon=False, loc="upper left")
    ax.set_title("두 라벨이 다른 리뷰 한 건마다 보정한 비율이 3.3%p 씩 바뀝니다")
    plt.show()

단계 1 의 MY_NEG 를 채운 뒤 위 셀을 실행하면 그림이 그려집니다

구간이 LLM 라벨 불만 비율 52.3% 를 포함하면, 30건으로는 편향이 있다고 단정할 수 없습니다. 그래도 결과는 보정한 비율과 구간으로 쓰고, 구간을 줄이려면 골드 표본을 늘립니다(단계 4).

**직접 해 보기: 결과 문장 쓰기**

위 출력으로 결과 문장을 `MY_SENTENCE` 에 씁니다. 보정한 비율은 골드 표본과 모델 · 프롬프트 버전이 바뀌면 달라지는 값이라, 아래 4가지가 없으면 어느 계산에서 나온 수인지 알 수 없습니다.

- 보정한 비율(%)
- 95% 구간
- 골드 표본 건수
- 모델과 프롬프트 버전: 불만 있음 라벨은 속성 라벨에서 구했으므로 프롬프트 버전은 aspect-v1 입니다. 모델과 버전은 「자료 준비」 셀 출력에 있습니다.

이론의 결과 문장과 같은 형식으로 씁니다. 「텀블러 리뷰 불만 비율 27%(95% 구간 12～42%). LLM 라벨 500건을 골드 표본 20건으로 보정한 비율. …」

In [ ]:
# 단계 3 · 직접 해 보기(실행 뒤): 결과 문장(보정한 비율 · 95% 구간 · 골드 표본 건수 · 모델과 프롬프트 버전)
MY_SENTENCE = ""

In [ ]:
# 단계 3 · 확인 셀: MY_GUESS 가 맞았는지와 MY_SENTENCE 의 4가지
# ac_ 로 시작하는 함수는 맨 위 설치 셀에서 정의합니다
NOT_YET = ac_state(None)[1]                                   # 빈 칸일 때의 안내(아직 채우지 않았습니다)
GOLD_N = re.compile(r"(?<![\d,.])30\s?건")                     # 골드 표본 건수(더 큰 건수의 끝자리는 빼고)


def check_ppi(guess, sentence):
    PPI = globals().get("PPI")
    g_ok, g_msg = ac_state(guess)
    s_ok, s_msg = ac_state(sentence)
    if g_msg == NOT_YET and s_msg == NOT_YET:
        print("아직 채우지 않았습니다. 위 셀의 MY_GUESS · MY_SENTENCE 를 채운 뒤 다시 실행합니다")
        return
    if PPI is None:
        print("단계 1 의 MY_NEG 를 채운 뒤 단계 3 의 계산 셀을 실행하고, 이 셀을 다시 실행합니다")
        return
    now, _ = parse_labels(globals().get("MY_NEG", ""))
    if now is None or "".join(map(str, now)) != PPI.get("neg"):
        print("단계 1 의 MY_NEG 가 바뀌었습니다. 단계 3 의 계산 셀을 다시 실행한 뒤 이 셀을 다시 실행합니다")
        return
    right = "높습니다" if PPI["adj"] > 0 else "낮습니다" if PPI["adj"] < 0 else "같습니다"
    versus = "LLM 라벨 불만 비율" + ("과 " if right == "같습니다" else "보다 ") + right
    word = {"높다": "높습니다", "낮다": "낮습니다", "같다": "같습니다"}.get(guess) if g_ok else None
    if g_msg == NOT_YET:
        print(f"MY_GUESS: {NOT_YET}")
    elif word is None:
        print('MY_GUESS: 고칠 것: "높다" · "낮다" · "같다" 가운데 하나로 적습니다')
    else:
        print(f"MY_GUESS: {'맞았습니다' if word == right else '아직 다릅니다'}. 나만 있음 {PPI['plus']}건 · LLM 만 있음 {PPI['minus']}건이라"
              f" 보정값이 {signed(PPI['adj'], 1)}%p 이고, 보정한 비율이 {versus}")
    if not s_ok:
        print(f"MY_SENTENCE: {s_msg}")
        return

    def found(value):                                         # 계산 셀의 출력처럼 소수 첫째 자리로 적었거나 이론의 예처럼 정수로 적었다
        return ac_has_value(sentence, value, digits=1) or ac_has_value(sentence, value, digits=0)

    est_ok = found(PPI["est"])
    ac_show("결과 문장 점검표", {                                   # 결과 문장의 4가지가 있는지
        f"보정한 비율 {PPI['est']:.1f}%": est_ok,
        f"95% 구간 {PPI['lo']:.1f}～{PPI['hi']:.1f}%": found(PPI["lo"]) and found(PPI["hi"]),
        "골드 표본 건수 30건": bool(GOLD_N.search(sentence)),
        "모델과 프롬프트 버전(Qwen3-4B · aspect-v1)": "qwen3-4b" in sentence.lower() and "aspect-v1" in sentence})
    if not est_ok and any(0 < v < 1 for v in ac_numbers(sentence)):   # 소수점 앞이 0 인 소수: 비율을 % 대신 소수로 적었다
        print("  비율을 % 값으로 적었는지 확인합니다(0.5 같은 소수가 아니라 50.0% 처럼)")


check_ppi(globals().get("MY_GUESS"), globals().get("MY_SENTENCE", ""))

아직 채우지 않았습니다. 위 셀의 MY_GUESS · MY_SENTENCE 를 채운 뒤 다시 실행합니다

<details class="lms-extra"><summary>자세히 — 예시 답</summary>

`MY_GUESS = "높다"`

`MY_SENTENCE = "리뷰 1,000건의 불만 비율 55.6%(95% 구간 42.3～72.3%). LLM 라벨 1,000건을 골드 표본 30건으로 보정한 비율. 모델 Qwen3-4B-Instruct-2507(버전 cdbee75f), 프롬프트 버전 aspect-v1."`

- 나만 있음 3건이 LLM 만 있음 2건보다 1건 많아 보정값이 +3.3%p 이고, 보정한 비율이 LLM 라벨 불만 비율 52.3% 보다 높습니다.
- 두 라벨이 다른 리뷰 한 건마다 보정한 비율이 1 ÷ 30 ≈ 3.3%p 씩 바뀝니다.
- 구간 길이가 30.0%p 로 넓습니다. 다른 30건을 골랐다면 두 방향의 건수가 달라져 보정값도 크게 달라질 수 있습니다.

</details>

> **[문제] 골드 표본으로 보정한 품질 부정 비율**
>
> 예를 들어 리뷰 1,000건의 LLM 라벨 품질 부정 비율이 36% 입니다. 골드 표본 100건에서 사람 라벨 품질 부정 비율은 32%, 같은 100건의 LLM 라벨 품질 부정 비율은 35% 입니다. 표본에서 구한 편향과 보정값, 보정한 비율을 차례로 적어 보세요.

## 단계 4 표본 크기

골드 표본을 늘리면 보정한 비율의 구간은 얼마나 짧아질까요? 별점 연습으로 확인합니다.

**별점 연습**은 별점이 모두 있는 리뷰 2만 건에서, 별점이 무작위 순서의 앞 n건에만 있다고 가정하고 보정한 비율을 구하는 연습입니다. 참값인 2만 건의 별점 부정 비율을 알고 있어서, 구한 값이 맞는지 확인할 수 있습니다. **별점이 있는 표본**은 이 연습에서 별점을 참값으로 쓰는 앞 n건입니다. 골드 표본과 달리 참값이 사람 라벨이 아니라 별점입니다.

- 보정한 비율 = 2만 건의 LLM 라벨 부정 비율 + 표본 n건의 (별점 − LLM 라벨) 평균
- 95% 구간: 표본 n건의 차이를 복원 추출로 2,000번 다시 골라 구합니다. 크기마다 시드 0 에서 새로 시작합니다.

> **[예측] 표본을 30건에서 480건으로 16배 늘리면 95% 구간 길이는 어떻게 될까요?**
>
> 1. 약 16분의 1
> 2. 약 4분의 1
> 3. 거의 그대로
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 단계 4: 별점 연습. 별점이 무작위 순서의 앞 n건에만 있다고 하고 보정한 비율과 95% 구간을 구합니다
SIZES = [30, 120, 480]
P_LLM20 = LLM20.mean()                                        # 2만 건의 LLM 라벨 부정 비율
SIZE_TABLE = []
for n in SIZES:
    diff = STAR20[:n] - LLM20[:n]                              # 별점이 있는 표본 n건의 (별점 − LLM 라벨)
    rng = np.random.default_rng(0)                             # 크기마다 같은 시드에서 시작
    boots = [P_LLM20 + diff[rng.integers(0, n, n)].mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    SIZE_TABLE.append({"n": n, "est": 100 * (P_LLM20 + diff.mean()), "lo": 100 * lo, "hi": 100 * hi,
                       "star_only": int((diff == 1).sum()), "llm_only": int((diff == -1).sum())})
print("표 3 · 별점이 있는 표본의 크기와 보정한 비율의 95% 구간 (%)")
for r in SIZE_TABLE:
    print(f"{r['n']}건: 보정한 비율 {r['est']:.2f}% · 95% 구간 {r['lo']:.2f}～{r['hi']:.2f}% · 구간 길이 {r['hi'] - r['lo']:#.3g}%p"
          f" · 별점만 부정 {r['star_only']}건 · LLM 라벨만 부정 {r['llm_only']}건")
print(f"2만 건: 별점 부정 비율 {rnd(100 * STAR20.mean(), 2):.2f}% · LLM 라벨 부정 비율 {100 * P_LLM20:.2f}%")

표 3 · 별점이 있는 표본의 크기와 보정한 비율의 95% 구간 (%)
30건: 보정한 비율 51.39% · 95% 구간 41.39～61.39% · 구간 길이 20.0%p · 별점만 부정 1건 · LLM 라벨만 부정 1건
120건: 보정한 비율 50.56% · 95% 구간 45.56～55.56% · 구간 길이 10.0%p · 별점만 부정 4건 · LLM 라벨만 부정 5건
480건: 보정한 비율 51.39% · 95% 구간 48.68～54.10% · 구간 길이 5.42%p · 별점만 부정 21건 · LLM 라벨만 부정 21건
2만 건: 별점 부정 비율 49.74% · LLM 라벨 부정 비율 51.39%

In [ ]:
# 단계 4 · 그림: 표본 크기마다 보정한 비율(점)과 95% 구간(가로선)
fig, ax = plt.subplots(figsize=(7, 3))
for y, r in enumerate(SIZE_TABLE):
    ax.hlines(y, r["lo"], r["hi"], color="#2251FF", linewidth=3)
    ax.plot(r["est"], y, "o", color="#051C2C")
ax.axvline(100 * STAR20.mean(), color="gray", linestyle="--", label="2만 건의 별점 부정 비율")
ax.set_yticks(range(len(SIZES)), [f"{n}건" for n in SIZES])
ax.set_ylim(-0.5, 3.0)
ax.set_xlabel("부정 비율(%)")
ax.legend(frameon=False, loc="upper left")
ax.set_title("그림 1 · 별점이 있는 표본의 크기와 보정한 비율의 95% 구간")
plt.show()

답은 둘째 보기, 약 4분의 1 입니다. 구간 길이는 30건에서 20.0%p, 480건에서 5.42%p 입니다. 표본을 4배로 늘릴 때마다 구간 길이가 절반쯤이 되므로, 16배면 4분의 1 쯤입니다.

- 30건과 480건은 두 방향의 건수가 같아 보정값이 0 이고, 보정한 비율이 LLM 라벨 부정 비율 51.39% 와 같습니다.
- 그래서 480건의 점은 참값인 별점 부정 비율 49.74% 와 1.65%p 떨어져 있습니다. 그래도 구간 48.68～54.10% 는 참값을 포함하므로, 보정한 비율은 점 하나가 아니라 구간으로 판단합니다.

**오차 한계**는 95% 구간 길이의 절반입니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/63/Margin_of_error_vs_sample_size_and_confidence_level.svg/960px-Margin_of_error_vs_sample_size_and_confidence_level.svg.png" alt="가로축 표본 크기 n 10 부터 10만, 세로축 오차 한계 0.1 부터 100 퍼센트인 로그 스케일 그래프. 신뢰 수준마다 오른쪽 아래로 내려가는 평행한 직선이 있고, 검은 점선이 95% 이다. n = 1000 에서 95% 는 3.1, 99% 는 4.1 이라고 화살표로 표시되어 있고, 오른쪽 위 작은 그림은 p 에 따른 보정 계수 곡선이다" width="560" height="420"/>
<p align="center"><em>▲ 가로축은 표본 크기 n, 세로축은 최대 오차 한계(±%)이고 모두 로그 스케일이다. 검은 점선이 95% 신뢰 수준이다. 참값만 쓴 비율의 선이라 n = 1,000 에서도 ±3.1%p 로, 표 3 의 480건 보정한 비율 ±2.71%p 보다 넓다 (출처: Wikimedia Commons, Margin of error vs sample size and confidence level.svg, Cmglee, CC BY-SA 4.0)</em></p>

**직접 해 보기: 골드 표본 건수 정하기**

운영팀이 리뷰 전체 부정 비율을 오차 한계 ±3%p 안팎으로 알고 싶어 합니다. 별점이 무작위 순서의 앞 n건에만 있다면 몇 건이 필요한지 표 3 으로 정합니다.

- `MY_SIZE`: 30 · 120 · 480 가운데 하나(정수)
- `MY_WHY`: 고른 이유 한 문장(문자열)

In [ ]:
# 단계 4 · 직접 해 보기: 오차 한계 ±3%p 안팎에 필요한 건수(30 · 120 · 480) · 고른 이유 한 문장
MY_SIZE = None
MY_WHY = ""

In [ ]:
# 단계 4 · 확인 셀: 오차 한계 ±3%p 안팎이 되는 가장 작은 크기와 MY_SIZE 를 비교합니다
# ac_ 로 시작하는 함수는 맨 위 설치 셀에서 정의합니다
NOT_YET = ac_state(None)[1]                                   # 빈 칸일 때의 안내(아직 채우지 않았습니다)
WHY_WORDS = re.compile(r"구간|오차|%p|±")                      # 구간 길이나 오차 한계로 든 이유


def check_size(size, why):
    z_ok, z_msg = ac_state(size, "number")
    w_ok, w_msg = ac_state(why)
    if z_msg == NOT_YET and w_msg == NOT_YET:
        print("아직 채우지 않았습니다. 위 셀의 MY_SIZE · MY_WHY 를 채운 뒤 다시 실행합니다")
        return
    if "SIZE_TABLE" not in globals():
        print("단계 4 의 첫 셀(표 3)을 먼저 실행한 뒤 이 셀을 다시 실행합니다")
        return
    half = {r["n"]: (r["hi"] - r["lo"]) / 2 for r in SIZE_TABLE}
    if z_msg == NOT_YET:
        print(f"MY_SIZE: {NOT_YET}")
    elif not z_ok or not isinstance(size, int) or size not in SIZES:
        print(f"고칠 것: MY_SIZE 는 {SIZES} 가운데 하나를 따옴표 없이 정수로 적습니다")
    else:
        right = min([n for n in SIZES if half[n] <= 3.0] or [max(SIZES)])
        print(f"MY_SIZE: {'맞았습니다' if size == right else '아직 다릅니다'}. 오차 한계가 ±3%p 안팎인 가장 작은 크기는 {right}건입니다")
        for n in SIZES:
            print(f"  {n}건: 오차 한계 ±{half[n]:#.3g}%p · 라벨 시간은 30건의 {n // 30}배")
    quoted = [x for n in SIZES for x in (half[n], 2 * half[n])]   # 오차 한계와 구간 길이: 이 값을 적었으면 부정문이어도 이유로 본다
    if not w_ok:
        print(f"MY_WHY: {w_msg}" + (". 고른 이유를 한 문장으로 적어 보세요" if w_msg == NOT_YET else ""))
    elif ac_claims(why, WHY_WORDS) or any(ac_has_value(why, x, digits=1) for x in quoted):
        print("MY_WHY: 구간 길이나 오차 한계로 이유를 들었습니다")
    else:
        print("MY_WHY: 구간 길이나 오차 한계로 이유를 들었는지 확인합니다")


check_size(globals().get("MY_SIZE"), globals().get("MY_WHY", ""))

아직 채우지 않았습니다. 위 셀의 MY_SIZE · MY_WHY 를 채운 뒤 다시 실행합니다

<details class="lms-extra"><summary>자세히 — 예시 답</summary>

`MY_SIZE = 480` · `MY_WHY = "120건은 오차 한계가 ±5.00%p 로 넓고, 480건은 ±2.71%p 로 ±3%p 보다 좁다"`

- 120건의 오차 한계는 ±5.00%p 로 ±3%p 보다 넓습니다. 480건은 ±2.71%p 입니다.
- 480건은 라벨 시간이 30건의 16배입니다. 기한 안에 그 시간을 쓸 수 없다면, ±5%p 로 알려도 되는지 운영팀과 먼저 정합니다.
- 불만 비율의 골드 표본은 이 별점 연습보다 구간이 길 수 있습니다. 단계 3 의 예시 답 30건은 구간 길이가 30.0%p 였습니다.
- 두 라벨이 다른 리뷰가 많을수록 구간이 깁니다. 단계 3 의 30건은 5건이 달랐고, 별점 연습의 30건은 2건이 달랐습니다.

</details>

## 단계 5 집단 · 범주마다 다른 재현율

LLM 라벨로 두 집단의 부정 비율 차이를 구해도 될까요? LLM 라벨로 구한 차이는 두 집단의 재현율 · 오탐률에 따라 실제 차이와 달라지므로, 집단마다 두 비율을 먼저 확인합니다.

두 집단은 리뷰 2만 건을 본문 길이의 중앙값(29자)으로 나눴습니다. 재현율과 오탐률은 참값(별점 극성)으로 구합니다.

- **재현율**: 별점 부정 리뷰 가운데 LLM 라벨도 부정인 비율
- **오탐률**: 별점 긍정 리뷰 가운데 LLM 라벨이 부정인 비율

<img src="https://upload.wikimedia.org/wikipedia/commons/2/26/Precisionrecall.svg" alt="왼쪽 반은 relevant elements, 오른쪽 반은 그 밖의 요소이고 가운데 원이 retrieved elements 인 그림. 아래에 Precision 과 Recall 을 원과 반원의 넓이로 그린 두 분수가 있다" width="340" height="618"/>
<p align="center"><em>▲ 정밀도와 재현율. 왼쪽 반 relevant elements 가 실제 양성, 가운데 원 retrieved elements 가 모델이 고른 것이다. 표 4 에서는 별점 부정이 왼쪽 반, LLM 라벨 부정이 가운데 원이다. 재현율은 왼쪽 반에서, 오탐률은 오른쪽 반에서 원 안에 있는 비율이다 (출처: Wikimedia Commons, Precisionrecall.svg, Walber, CC BY-SA 4.0)</em></p>

In [ ]:
# 단계 5: 리뷰 길이 두 집단의 재현율 · 오탐률과 부정 비율(참값은 별점 극성)
print("표 4 · 리뷰 길이 두 집단의 재현율 · 오탐률과 부정 비율")
for g in GROUPS:
    print(f"{g['name']} {g['n']:,}건: 재현율 {g['recall']:.3f} · 오탐률 {g['fpr']:.3f}"
          f" · 별점 부정 {g['star']:.1f}% · LLM 라벨 부정 {g['llm']:.1f}%")
    print(f"  놓친 부정 {g['miss']:.1f}%p · 잘못 찾은 부정 {g['fp']:.1f}%p")
gap_star = GROUPS[1]["star"] - GROUPS[0]["star"]
gap_llm = GROUPS[1]["llm"] - GROUPS[0]["llm"]
print(f"두 집단의 차이(긴 − 짧은): 별점 {gap_star:.1f}%p · LLM 라벨 {gap_llm:.1f}%p")

표 4 · 리뷰 길이 두 집단의 재현율 · 오탐률과 부정 비율
짧은 리뷰(29자 이하) 10,151건: 재현율 0.891 · 오탐률 0.082 · 별점 부정 42.6% · LLM 라벨 부정 42.6%
  놓친 부정 4.7%p · 잘못 찾은 부정 4.7%p
긴 리뷰(29자 초과) 9,849건: 재현율 0.971 · 오탐률 0.116 · 별점 부정 57.1% · LLM 라벨 부정 60.4%
  놓친 부정 1.7%p · 잘못 찾은 부정 5.0%p
두 집단의 차이(긴 − 짧은): 별점 14.5%p · LLM 라벨 17.8%p

In [ ]:
# 단계 5 · 그림: 리뷰 길이 두 집단의 부정 비율(별점 · LLM 라벨)
x = np.arange(len(GROUPS))
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(x - 0.2, [g["star"] for g in GROUPS], width=0.4, color="gray", label="별점")
ax.bar(x + 0.2, [g["llm"] for g in GROUPS], width=0.4, color="#2251FF", label="LLM 라벨")
ax.set_xticks(x, ["짧은 리뷰", "긴 리뷰"])
ax.set_ylabel("부정 비율(%)")
ax.legend(frameon=False)
ax.set_title("그림 2 · 리뷰 길이 두 집단의 부정 비율: 별점과 LLM 라벨")
plt.show()

놓친 부정은 별점은 부정인데 LLM 라벨이 긍정인 리뷰이고, 잘못 찾은 부정은 별점은 긍정인데 LLM 라벨이 부정인 리뷰입니다. 표 4 의 두 값은 그 집단의 리뷰 수로 나눈 비율입니다.

```text
LLM 라벨 부정 비율 = 별점 부정 비율 − 놓친 부정 + 잘못 찾은 부정
```

- 긴 리뷰의 놓친 부정은 (1 − 재현율 0.971) × 별점 부정 57.1% ≈ 1.7%p 입니다. 잘못 찾은 부정은 오탐률 0.116 × 별점 긍정 42.9% ≈ 5.0%p 로 더 많아, LLM 라벨이 3.3%p 높습니다.
- 짧은 리뷰는 재현율이 더 낮아도 놓친 부정과 잘못 찾은 부정이 둘 다 4.7%p 라 상쇄됩니다.

> **[문제] 리뷰 길이 두 집단의 부정 비율 차이**
>
> 「표 4 · 리뷰 길이 두 집단의 재현율 · 오탐률과 부정 비율」과 「그림 2 · 리뷰 길이 두 집단의 부정 비율: 별점과 LLM 라벨」에서 긴 리뷰와 짧은 리뷰의 부정 비율 차이로 무엇을 쓸까요?
>
> 1. 별점의 차이: 두 집단의 재현율 · 오탐률이 다르다
> 2. LLM 라벨의 차이: 리뷰가 2만 건이라 믿을 만하다
> 3. 쓰지 않는다: 두 라벨로 구한 차이가 서로 다르다

리뷰 극성은 범주가 둘뿐이지만, 범주가 많은 분류기도 범주마다 재현율을 따로 구해야 합니다. 드문 범주를 놓치면 그 범주의 건수가 실제보다 적게 집계되기 때문입니다.

이 분류기는 민원 사건제목의 명사로 TF-IDF 를 만들고, 로지스틱 회귀로 상담이유 16종 가운데 하나를 고릅니다. 민원 표본 10만 건 가운데 앞 8만 건으로 학습하고 뒤 2만 건으로 평가했습니다. 드문 두 상담이유는 부당채권추심과 무능력자계약(미성년자처럼 혼자서는 계약할 수 없는 사람이 맺은 계약)입니다.

- 정확도는 최빈 범주 베이스라인 정확도(늘 가장 흔한 상담이유만 답할 때의 정확도)와 비교합니다.
- 드문 범주를 놓쳤는지는 범주별 재현율로 확인합니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_confusion_matrix_002.png" alt="Normalized confusion matrix 제목의 3×3 표. 행 True label 과 열 Predicted label 은 모두 setosa · versicolor · virginica 다. 대각선은 1 · 0.19 · 1 이고 versicolor 행의 virginica 열이 0.81 이다" width="480" height="360"/>
<p align="center"><em>▲ 행마다 나눈 혼동행렬. 행 True label 은 실제, 열 Predicted label 은 예측이다. 대각선 값이 그 범주의 재현율이다. 표 5 의 두 재현율은 상담이유 16종 표에서 부당채권추심 행과 무능력자계약 행의 대각선 값이다 (출처: scikit-learn 문서, Confusion matrix 예제, BSD-3-Clause)</em></p>

In [ ]:
# 단계 5: 민원 상담이유 분류기의 정확도와 드문 두 상담이유의 재현율(평가 민원 2만 건)
print("표 5 · 상담이유 분류기의 정확도와 드문 상담이유의 재현율 (평가 민원 2만 건)")
print(f"정확도 {CLF['acc']:.3f} · 최빈 범주 베이스라인 정확도 {CLF['base']:.3f}(늘 「{CLF['top']}」만 답할 때)")
for r in CLF["rare"]:
    print(f"{r['name']}: 평가 {r['n']}건 · 분류기가 찾은 건수 {r['hit']}건 · 재현율 {r['hit'] / r['n']:.3f}")
print("이론의 예와 다른 민원 표본(평가 2만 건)이라 수가 이론과 다릅니다")

표 5 · 상담이유 분류기의 정확도와 드문 상담이유의 재현율 (평가 민원 2만 건)
정확도 0.553 · 최빈 범주 베이스라인 정확도 0.268(늘 「계약해제.해지/위약금」만 답할 때)
부당채권추심: 평가 20건 · 분류기가 찾은 건수 3건 · 재현율 0.150
무능력자계약: 평가 22건 · 분류기가 찾은 건수 1건 · 재현율 0.045
이론의 예와 다른 민원 표본(평가 2만 건)이라 수가 이론과 다릅니다

In [ ]:
# 단계 5 · 그림: 상담이유 16종의 재현율(평가 건수가 많은 차례, 파랑은 드문 두 상담이유)
by = sorted(CLF["by_class"], key=lambda r: -r["n"])
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([f"{r['name']}({r['n']:,}건)" for r in by], [r["hit"] / r["n"] for r in by],
        color=["#2251FF" if r["name"] in ("부당채권추심", "무능력자계약") else "gray" for r in by])
ax.invert_yaxis()
ax.set_xlabel("재현율")
ax.set_title("평가 건수가 적은 상담이유는 재현율이 대체로 낮습니다")
plt.show()

> **[문제] 분류기 라벨로 드문 상담이유 집계하기**
>
> 동료가 이 분류기로 부당채권추심 민원 건수를 매달 집계하자고 합니다. 「표 5 · 상담이유 분류기의 정확도와 드문 상담이유의 재현율 (평가 민원 2만 건)」을 근거로 어떻게 답할까요?
>
> 1. 집계한다: 정확도가 절반을 넘는다
> 2. 집계한다: 정확도가 베이스라인보다 높다
> 3. 집계하지 않는다: 그 상담이유의 재현율이 낮다

## 결론

In [ ]:
# 결론 · 그림: 리뷰 1,000건의 불만 비율, LLM 라벨과 골드 표본 30건으로 보정한 비율(95% 구간)
PPI = globals().get("PPI")
p_llm = 100 * LABELS_1000["llm_neg"].mean()
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.plot(p_llm, 1, "o", color="gray")
title = "단계 1 의 MY_NEG 를 채우면 보정한 비율도 그려집니다"
if PPI is not None:
    ax.hlines(0, PPI["lo"], PPI["hi"], color="#2251FF", linewidth=3)
    ax.plot(PPI["est"], 0, "o", color="#051C2C")
    title = f"보정한 비율의 95% 구간이 LLM 라벨 {p_llm:.1f}% 를 " + ("포함합니다" if PPI["lo"] <= p_llm <= PPI["hi"] else "벗어납니다")
ax.set_yticks([0, 1], ["보정한 비율(골드 표본 30건)", "LLM 라벨(1,000건)"])
ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("불만 비율(%)")
ax.set_title(title)
plt.show()

<div class="lms-summary">

**정리**

- 리뷰 2만 건은 κ 0.84 인데도 LLM 라벨 부정 비율이 별점보다 1.65%p 높았습니다. 리뷰 전체 부정 비율은 별점 49.74% 를 씁니다.
- 불만 비율은 LLM 라벨 1,000건의 52.3% 를 그대로 쓰지 않고, 골드 표본 30건으로 보정한 비율과 95% 구간으로 씁니다.
- 골드 표본을 4배로 늘리면 보정한 비율의 구간 길이가 절반쯤이 됩니다.
- 리뷰 길이 두 집단은 재현율 · 오탐률이 달라, 두 집단의 차이는 별점으로 구한 14.5%p 를 씁니다.
- 부당채권추심은 재현율이 0.150 이라, 분류기 라벨로 그 건수를 집계하지 않습니다.

**할 일**

- 골드 표본은 무작위로 고르고, LLM 라벨을 보지 않고 라벨링합니다
- 결과 문장은 보정한 비율 · 95% 구간 · 골드 표본 건수 · 모델과 프롬프트 버전 4가지로 씁니다
- 구간이 넓으면 골드 표본을 늘립니다

LLM 라벨로 계산한 비율은 그대로 쓰지 않고, 참값이 있는 표본으로 확인하고 보정한 뒤에 씁니다.

</div>

> **[과제] 실습 제출: 골드 표본 30건의 라벨과 보정한 불만 비율**
>
> 직접 해 보기 5가지를 채우고 확인 셀을 실행한 상태로 노트북을 제출합니다. ① 단계 1 의 MY_NEG ② 단계 2 의 MY_MORE · MY_USE ③ 단계 3 의 MY_GUESS ④ 단계 3 의 MY_SENTENCE ⑤ 단계 4 의 MY_SIZE · MY_WHY 를 채웁니다. 단계 1 은 내 라벨이 LLM 라벨과 달라도 확인 셀이 결과를 출력했으면 점수를 받습니다.
>
> 마감: 2026-11-11T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 단계 1 | MY_NEG 30자를 모두 적었고 확인 셀이 일치율 · κ · 두 방향 건수를 출력했다(LLM 라벨과 달라도 점수) | 25 |
> | 단계 2 | MY_MORE · MY_USE 를 채웠고 확인 셀이 두 변수의 결과를 출력했다 | 25 |
> | 단계 3 | MY_GUESS · MY_SENTENCE 를 채웠고 확인 셀이 예측 결과와 결과 문장 점검표를 출력했다 | 25 |
> | 단계 4 | MY_SIZE · MY_WHY 를 채웠고 확인 셀이 크기마다 오차 한계를 출력했다 | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다. 비정형 데이터(자연어) 골드 표본과 LLM 라벨 검증 실습을 마쳤습니다</h2>
<div style="color:#656565">리뷰 30건을 직접 라벨링해 LLM 라벨과 비교하고, 내 라벨로 리뷰 1,000건의 불만 비율을 보정했습니다. 별점이 있는 표본의 크기에 따라 구간이 얼마나 짧아지는지도 확인했습니다.</div>
</div>